# Reproducing the results of MacInnis et. al. (2026)

This notebook will show you how to reproduce the results presented in [**TODO:LINK2PAPER**](https://arxiv.org/abs/XXXX.XXXXX) by applying the extragalactic foreground (FG) cleaning procedure described in that work to [ultrahigh-resolution simulations](https://github.com/CMB-HD/hdsims) of the lensed CMB, thermal and kinetic Sunyaev-Zel'dovich effect (tSZ and kSZ, respectively), the cosmic infrared background (CIB), radio galaxies, and instrumental noise at 90, 148, 219, and 277 GHz on a 100 square degree patch of the sky. 





---

The notebook is organized into different parts, summarized below:

- **Part 1**: This is where you will define or change any variables in the notebook, such as:
  - Paths to directories for the simulations, FG cleaning output, etc.
  - The number of MPI processes to use when running the python scripts we provide
  - Whether you would like to reproduce the parameter tables and plots of MacInnis et. al. 2026 (see below)

- **Part 2**: Save the simulations used for FG cleaning 

- **Part 3**: Run the FG cleaning procedure 
  - Apply matched filters to the maps to iteratively detect, measure, and remove point sources (CIB and radio galaxies) and tSZ clusters from the maps at each frequency
    - The catalogs of the sources and clusters that were removed from the maps, as well as the maps themselves after FG cleaning, will be saved (along with additional, intermediate output files); we provide Python methods to load these files for you.
  - Match the catalogs of detected sources and clusters to the true catalogs of all sources and clusters in the maps
  - Take the power spectra of the FG-cleaned maps

- **Part 4**: FG cleaning results
  - Compare the power spectra of the FG-cleaned maps to the files provided with the [hdMockData](https://github.com/CMB-HD/hdMockData) repository
  - Reproduce the plots in MacInnis et. al. of the FG-cleaned maps and power spectra, and the source and cluster recovery (Figures 1, 2, and 6 - 11)
  - Optionally, reproduce the parameter results in MacInnis et. al. (Tables 4 - 6 and Figures 12, 13)
    - **Note**: For this step, you will also need to install [hdfisher](https://github.com/CMB-HD/hdfisher) and [getdist](https://getdist.readthedocs.io) (in addition to the requirements of `hdfgclean`).

After "Part 1", you do not need to modify any of the notebook cells. As you run the cells in parts 2 and 3 of this notebook, instructions will be printed out with the commands you must run outside of this notebook (using the bash and python scripts we provide) before running subsequent notebook cells. If you choose to reproduce the parameter results and to calculate your own Fisher matrices, this must also be done outside of this notebook; the instructions will be provided in the "Parameters" section of "Part 4".

---

# Part 1 : 

Below, we import the packages required to run this notebook; from the `hdfgclean` package, we import two modules:
- `hdfgclean.py` contains the main `HDFGClean` class, used to run the FG cleaning and load in the resulting maps and catalogs
- `hdfgclean_utils.py` contains functions that are only used in this notebook to print out instructions, and compare the power spectra of the FG-cleaned maps to the files provided with the [hdMockData](https://github.com/CMB-HD/hdMockData) repository

In [ ]:
import os 
from hdfgclean import hdfgclean, hdfgclean_utils

In the cell below, you **must** provide the paths to:
- your `hd_sims_dir` where the simulations will be saved (see [hdsims](https://github.com/CMB-HD/hdsims) for more information)
- an `output_dir` where the output from FG cleaning will be saved

**Note** that you will need about 70 GB of space to save the simulations, and about 45 GB to save the FG cleaning output.

In [ ]:
hd_sims_dir = 
output_dir = 

If you have moved (or made a copy of) this notebook into a different directory, provide the path to the `hdfgclean` repository (i.e., the directory that contains the readme file) below; otherwise, you can leave this cell unchanged:

In [ ]:
hdfgclean_repo_dir = None

**Using MPI is strongly recommended** when running the FG cleaning in Part 3 (also see the MPI section of the `hdfgclean` "[README](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#using-mpi-strongly-recommended)" file). 

Below, you may change `num_mpi_processes_fgclean` (default is `25`) and `num_mpi_processes_spectra` (default is `2`), the number of MPI processes to use for the FG cleaning on the smaller patches or the power spectra of the full maps, respectively. If you're not using MPI, set each to `1`; however, in this case, we recommend running FG cleaning on a single smaller patch with the `example_2x2.ipynb` notebook we provide. (**TODO**)

- The input maps will be divided into 25 smaller patches, and the FG cleaning procedure can be run on each patch simultaneously, so we recommend running the FG cleaning with 25 MPI processes (one per patch), *and* running on multiple compute nodes.
  - While testing on the Stony Brook University [SeaWulf](https://rci.stonybrook.edu/HPC/understanding-SeaWulf) cluster, we used five nodes in the `long-96core` [queue](https://rci.stonybrook.edu/HPC/faqs/seawulf-queues) to FG clean the maps, which took about five hours.
  - (Note that, without MPI, the same FG cleaning run would have take about 5 hours $\times$ 25 patches = 125 hours total!)
- After FG cleaning the maps, we will match the catalogs of detected sources and clusters to the catalogs of true sources and clusters in the maps; this can also be done simultaneously on each patch.
  - This step requires less memory, so we ran it on a single node in the `short-96core` queue, which took about 1.5 hours.
- Then we take the power spectra of the full 100 square degree maps; this requires much more memory, so we recommend using fewer MPI processes for this step.
  - We used two MPI processes on a single node in the `hbm-short-96core` queue, which took about an hour.
  
If you are running the FG cleaning by submitting jobs on a cluster using [slurm](https://slurm.schedmd.com) (e.g. with the [sbatch](https://slurm.schedmd.com/sbatch.html) command), you can use the `--dependency` option to ensure that the first step (FG cleaning the maps) is completed before trying to match catalogs or take power spectra; see [here](https://hpc.nih.gov/docs/job_dependencies.html) for examples

In [ ]:
num_mpi_processes_fgclean = 25
num_mpi_processes_spectra = 2

If you would also like to reproduce the parameter results of MacInnis et. al. (2026), you must set `reproduce_param_results=True` below, and you may change any of the other variables defined in this cell. If `reproduce_param_results=False` (the default, due to the additional required packages), the other variables will be ignored. 

- If you would also like to calculate the Fisher matrices yourself, set `calculate_fisher = True` below; otherwise, if `calculate_fisher = False`, we will load in pre-computed Fisher matrices to print out and plot the parameter errors.
- If `calculate_fisher = True`, the Fisher matrices will be saved in sub-directories within the `fisher_dir`. By default, the `fisher_dir` will be the same directory as this notebook; you may define a different `fisher_dir` below.
- If `calculate_fisher = True`, you may use MPI by increasing `num_mpi_processes_fisher`, the number of MPI processes used for the calculation of the numerical derivatives of the theory power spectra with respect to the parameters. 
  - For reference, we found that the calculation took about a half hour without MPI on a single node of the `hbm-short-96core` [SeaWulf queue](https://rci.stonybrook.edu/HPC/faqs/seawulf-queues); it took about 10 minutes longer using 12 MPI processes.
- If you do *not* want to use LaTeX when printing out parameter tables in the notebook (e.g., for parameter names), set `use_latex=False`.
- If you do *not* want to save the parameter triangle plots that will be made, set `save_param_plots=False`.

In [ ]:
reproduce_param_results = False
calculate_fisher = False
fisher_dir = os.getcwd()
num_mpi_processes_fisher = 1
use_latex = True # only for printing out tables
save_param_plots = True

---

# Part 2: Save the simulations

In the following two cells, we will print out intructions to download the HD simulations being FG-cleaned from LAMBDA, and to generate the smaller set of maps used to quantify the noise in the matched filter calculation; see the `hdfgclean` "[README](https://github.com/CMB-HD/hdfgclean/blob/main/README.md#before-using-hdfgclean)" file for more information.

In [ ]:
hdfgclean_utils.print_instructions_to_download_hdsims(hd_sims_dir, hdfgclean_repo_dir=hdfgclean_repo_dir)

In [ ]:
hdfgclean_utils.print_instructions_to_generate_noise_sims(hd_sims_dir, hdfgclean_repo_dir=hdfgclean_repo_dir, output_dir=output_dir)

After following the instructions above, run the cell below to make sure the files were saved correctly; if they weren't, follow the instructions that are printed out:

In [ ]:
# check if the simulation files have been saved:
hdsims_are_saved = hdfgclean_utils.hdsims_files_are_saved(hd_sims_dir)
maps_for_noise_are_saved = hdfgclean_utils.noise_map_files_are_saved(hd_sims_dir)
sim_files_saved = hdsims_are_saved and maps_for_noise_are_saved
if sim_files_saved:
    print("You may proceed: all necessary files have been saved.")
else:
    # print out the instructions again:
    if not hdsims_are_saved:
        hdfgclean_utils.print_instructions_to_download_hdsims(hd_sims_dir, hdfgclean_repo_dir=hdfgclean_repo_dir)
    if not maps_for_noise_are_saved:
        hdfgclean_utils.print_instructions_to_generate_noise_sims(hd_sims_dir, output_dir=output_dir, hdfgclean_repo_dir=hdfgclean_repo_dir)
    print(f"\nThen, re-run this cell.")

---

# Part 3: Running FG cleaning

Below, we will save a `.yaml` configuration file with your `hd_sims_dir` and `output_dir`, which will be used to initialize the `HDFGClean` class. All other (optional) keyword arguments that can be passed to the `HDFGClean` class are set to the values used in MacInnis et. al. (2026) by default.

Then, we will print out instructions to run the FG cleaning with the `run_hdfgclean` method of `HDFGClean`, initialized with your configuration file; this will be done by running the provided `reproduce_10x10.py` python script.

In [ ]:
# save a configuration file to run the FG cleaning using `reproduce_10x10.py`
config_file = os.path.join(output_dir, 'hdfgclean.yaml')    
if not os.path.exists(config_file):
    hdfgclean.HDFGClean.save_config(config_file, hd_sims_dir, output_dir=output_dir)

In [ ]:
# print out instructions to run `reproduce_10x10.py` with this config file:
hdfgclean_utils.print_instructions_to_reproduce_10x10(config_file, hdfgclean_repo_dir=hdfgclean_repo_dir, 
                                                      num_mpi_processes_fgclean=num_mpi_processes_fgclean, 
                                                      num_mpi_processes_spectra=num_mpi_processes_spectra)

---

# Part 4: FG cleaning results

Below, we initialize the `HDFGClean` class. 

Note that, if this is the first time initializing `HDFGClean` with this `config_file`, the maps that are input to the FG cleaning procedure will be loaded in and saved: these maps are the beam-convolved sum of the lensed CMB, kSZ, tSZ, CIB, and radio maps, plus the instrumental noise maps; each map is about 2 GB. (However, you shouldn't have to worry about this if you've followed the instructions above!)

In [ ]:
hdfgcleanlib = hdfgclean.HDFGClean.from_config(config_file)

Now we will make sure everything has been saved:

In [ ]:
fgclean_files_saved = hdfgclean_utils.all_fgclean_files_are_saved(hdfgcleanlib, config_file, 
                                                                  num_mpi_processes_fgclean=num_mpi_processes_fgclean, 
                                                                  num_mpi_processes_spectra=num_mpi_processes_spectra)

---

Below, we compare the power spectra of your 90 and 148 GHz FG-cleaned maps to the following files that are provided with [hdMockData](https://github.com/CMB-HD/hdMockData):
- The power spectrum of residual CIB and radio sources at 90 and 148 GHz
- The residual tSZ power spectrum at 90 and 149 GHz
- The power spectrum of instrumental noise + kSZ + residual tSZ + residual CIB and radio sources ("FG + noise") at 90 and 148 GHz, and the coadded 90+148 GHz FG + noise power spectrum

In [ ]:
if fgclean_files_saved:
    hdfgclean_utils.compare_10x10_spectra(hdfgcleanlib)

---

## Plots

In the following cells, we will plot Figures 1, 2, and 6 - 11 (not necessarily in that order) of MacInnis et. al. 2026. In Figures 6, 7, 10, and 11, a smaller four-square-degree region at the center of the maps is used to make the plots; the remaining plots use the full 100-square-degree maps.


We provide a brief summary of each plot and, when applicable, point out the relevant methods of the `HDFGClean` class used to access the results shown. You may refer to MacInnis et. al. (2026) or the documentation of the `HDFGClean` methods (or the inherited methods of its parent classes), respectively, for further details. You are not required to read this extra information before proceeding.


However, **note** that if you set both `reproduce_param_results = True` and `calculate_fisher = True` in "Part 1" above, then you must read the instructions under the "Parameters" section below to calculate the Fisher matrices.

### (Figure 6) Measured point source fluxes

The _left panel_ compares the measured and true fluxes of 90 GHz CIB+radio point sources that were detected with SNR $\geq$ 4 and matched to a true source.
- The `match_to_true_sources_catalog` method (defined in the `HDFGCleanResults` class of the `hdfgclean_results.py` module) returns the catalog of matched sources at a given frequency, which has columns for the measured and true flux of each source. 
  - **Note** that, if the catalog is not already saved, the matching will be done when you call this method; if you have not run the FG cleaning yet, this means that the full FG cleaning will be run first. Therefore, you should always ensure that the catalogs are saved by, e.g., running the `run_hdfgclean.py` python script provided in the `hdfgclean` repository (we did this by running the `reproduce_10x10.py` script earlier).


The _right panel_ shows the measured 148 GHz and 277 GHz fluxes of CIB sources used to calculate the average 277-to-148 CIB spectral index.
- The `get_spectral_index` method returns the calculated CIB or radio spectral index between two frequencies, and the `get_spectral_index_catalog` method returns a catalog of sources used to calculate the index; both are defined in the `FGClean` class in the `fgclean.py` module.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.flux_measurement_plot()

### (Figure 7) Statistics of detected point sources



The _upper panels_ show, for 90 and 148 GHz (left and right panels, respectively), the number of bright sources detected with SNR $\geq$ 4 at the given frequency (yellow), dim sources detected at a different frequency (pink), and sources not detected (blue), all as a function of the true source flux; the number of false detections (as a function of measured flux) are shown in red.
 - The `match_to_true_sources_catalog` method (defined in the `HDFGCleanResults` class of the `hdfgclean_results.py` module) returns catalogs at a given frequency  of matched sources (with columns for the measured and true flux of each source), false detections, and true sources that were not detected; see the note under Figure 6 about this method.


The _lower panels_ show the power spectra of the CIB + radio maps at 90 and 148 GHz (left and right panels, respectively) before any FG cleaning (blue dotted), after subtracting bright sources detected with SNR $\geq$ 4 (yellow), and after subtracting all (bright and dim) sources (pink).

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.sources_plot()

### (Figure 9) SZ cluster masses and redshifts

The true mass and redshift of all SZ clusters in the map that were detected with SNR $\geq$ 4 (red) and those that were not detected (blue).
- The `match_to_true_clusters_catalog` method (defined in the `HDFGCleanResults` class of the `hdfgclean_results.py` module) returns catalogs of true clusters that were detected, and those that were not detected. The note under Figure 6 also applies to this method.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.cluster_mass_vs_redshift_plot()

### (Figure 8) SZ cluster completeness

The completeness of detected SNR $\geq$ 4 clusters per mass and redshift bin.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.cluster_completeness_per_mass_redshift_plot()

### (Figure 10) Maps before and after FG cleaning

A four-square-degree region of the maps before (first column) and after (second column) FG cleaning. The first row shows 90 GHz CMB + FG (kSZ, tSZ, CIB, radio) maps. The second row shows the 90 GHz maps after subtracting out the frequency-independent CMB and kSZ for clarity, and the third row shows the corresponding maps at 148 GHz.

- The `get_sim` method of the `HDFGClean` class is used to obtain each map. This is the same method as the one provided by the `HDSims` class (in the `hdsims.py` module of the `hdsims` package). 
  - It is originally defined in the `HDSimsGen` class of the `hdsimsgen.py` module of the `hdsims` package. 
  - The `HDSims.get_sim` method is overridden in the `HDFGCleanMaps` class of the `hdfgclean_maps.py` module. The optional keyword arguments `subtract_sources` and `subtract_clusters` (both `False` by default) have been added; if `True`, the detected sources or clusters, respectively, are subtracted from the map.
    - Note that if you pass `subtract_sources=True` or `subtract_clusters=True`, the full FG cleaning procedure will be run if it has not already been done. Therefore, you should always ensure that the maps have been FG cleaned by, e.g., running the `run_hdfgclean.py` python script provided in the `hdfgclean` repository (we did this by running the `reproduce_10x10.py` script earlier).
    
It may take a few minutes to make this plot.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_fgcleaned_maps_with_cmb()

### (Figure 11) Maps of the tSZ, CIB, and radio sources

A four-square-degree region of the tSZ + CIB + radio maps (i.e., with the CMB and kSZ subtracted out for clarity) at 90 and 148 GHz (top and bottom rows, respectively). The first and last columns show the maps before and after FG cleaning, respectively. The middle column shows maps of the measured CIB + radio point sources and tSZ clusters; these maps are subtacted from the maps in the first column to produce the maps in the last column.

- The maps of the measured point sources or clusters at a given frequency can be obtained using the `get_map_of_subtracted_sources` or `get_map_of_subtracted_clusters` methods, respectively; both are defined in the `FGClean` class of the `fgclean.py` module.
  - The note under Figure 10 also applies to these methods.
  
It may take a few minutes to make this plot.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_fgcleaned_maps_without_cmb()

### (Figure 2) Simulation-based residual FG power spectra at 90 and 148 GHz

Residual CIB+radio (blue), tSZ (green), and total FG + noise (dark red) power spectra of the 90 and 148 GHz (left and right panels, respectively) maps after FG cleaning; these are the same power spectra that you compared to the precomputed files earlier in this part. The previous estimate (from [MacInnis and Sehgal (2024)](https://arxiv.org/abs/2405.12220)) of the CMB-HD total FG + noise power spectra is shown in light red.

- The `get_sim_power` method of `HDFGClean` can be used to obtain the simulation-based power spectra. This is the same method as the one provided by the `HDSims` class (in the `hdsims.py` module of the `hdsims` package). 
  - It is originally defined in the `HDSimsSpectra` class of the `hdsims_spectra.py` module of the `hdsims` package. 
    - Note that the power spectra (and the inverse mode-coupling matrix, if necessary) will be calculated when you call this method, if it has not been saved already.
  - The `HDSims.get_sim_power` method is overridden in the `HDFGCleanSpectra` class of the `hdfgclean_spectra.py` module. The optional keyword arguments `subtract_sources` and `subtract_clusters` (both `False` by default) have been added; if `True`, the detected sources or clusters, respectively, are subtracted from the map. You may also pass `mask=True` (default is `False`) to apply the mask to the maps before taking their power spectra.
    - Note that if you pass `subtract_sources=True`, `subtract_clusters=True`, or `mask=True`, the full FG cleaning procedure will be run if it has not already been done. Therefore, you should always ensure that the maps have been FG cleaned and the power spectra has been saved by, e.g., running the `run_hdfgclean.py` python script provided in the `hdfgclean` repository (we did this by running the `reproduce_10x10.py` script earlier).

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_fgcleaned_spectra()

### (Figure 1) Coadded 90 & 148 GHz simulation-based residual FG + noise power spectrum

The coadded 90 & 148 GHz power spectra of the residual FGs (kSZ, tSZ, CIB, radio) + instrumental noise. The simulation-based power spectrum is shown in dark red, and the previous estimate of MacInnis and Sehgal (2024) is shown in light red. The former is the same simulation-based power spectrum you compared to the precomputed files earlier in this part. 

- The simulation-based coadded residual FG + noise power spectrum is returned by the `get_coadded_fgcleaned_sim_power` method, defined in the `HDFGCleanSpectra` class of the `hdfgclean_spectra.py` module. The same note written under Figure 2 applies to this method.

In [ ]:
if fgclean_files_saved:
    hdfgcleanlib.plot_coadded_fgcleaned_spectra()

---

## Parameters

The following cells will only run if you have set `reproduce_param_results = True` in "Part 1".

Below, we import the `hdfgclean_params` module, which is only used in this notebook (nowhere else within the `hdfgclean` package). This module contains functions to load in either pre-computed Fisher matrices or the ones that you will calculate, and use them to print out tables of parameter error bars and make the parameter triangle plots. Note that the priors described in MacInnis et. al. (2026) have already been applied to all Fisher matrices.

If you have set `calculate_fisher_matrices = True` above, you must follow the instructions that will be printed out to run the `calculate_fisher.py` python script before proceeding.
- Note that the simulation-based, coadded residual FG + noise *TT* power spectrum is used to calculate the CMB lensing noise curves and the CMB-HD covariance matrices, which are used in the calculation of the delensed CMB power spectra and the Fisher matrices, respectively. You have already confirmed that your simulation-based, coadded residual FG + noise power spectrum matches the power spectrum provided in the `hdMockData` repository (as version `v1.2`). 

In [ ]:
if reproduce_param_results:
    from hdfgclean import hdfgclean_params # only used in this section of this notebook
    # needed to display the parameter plots in this notebook:
    import matplotlib
    %matplotlib inline
    
    # print out instructions to calculate new Fisher matrices, if applicable:
    fisher_output_dir = fisher_dir if calculate_fisher else None
    hdfgclean_params.print_fisher_instructions(fisher_output_dir, 
                                               hdfgclean_repo_dir=hdfgclean_repo_dir, 
                                               num_mpi_processes_fisher=num_mpi_processes_fisher)
    # check if the Fisher matrices have been saved:
    fisher_matrices_saved = hdfgclean_params.fisher_matrices_saved(fisher_output_dir=fisher_output_dir)

else:
    print("There is nothing left to do in this notebook!")
    fisher_matrices_saved = False


In the following cell, if you have calculated your own Fisher matrices, we print out the ratio of your 1-sigma parameter error bars to the values in each column of Table 4 of MacInnis et. al. 2026 (see below for a description of each column). 

In [ ]:
if calculate_fisher and fisher_matrices_saved:
    hdfgclean_params.compare_param_errors_to_precomputed(fisher_output_dir, use_latex=use_latex)

Below, we print out Tables 4, 5, and 6 of MacInnis et. al. (2026). In all cases, we combine CMB-HD delensed (as opposed to lensed) CMB $TT$, $TE$, $EE$, $BB$ power spectra and the CMB lensing $\kappa\kappa$ power spectrum with the mock DESI BAO data from [MacInnis et. al. (2023)](https://arxiv.org/abs/2309.03021).
- Table 4: 1-sigma parameter error bars for 
  - three sets of varied parameters: 
    - $\Lambda$CDM + $N_\mathrm{eff}$ + $\sum m_\nu$: vary the six $\Lambda$CDM parameters, the effective number of relativistic species, and the sum of neutrino masses (in eV)
    - $\Lambda$CDM + $N_\mathrm{eff}$ + $\sum m_\nu$ + $\log T_\mathrm{AGN}$: additionally vary $\log_{10}(T_\mathrm{AGN}/\mathrm{K})$, which characterizes the strength of baryonic feedback
    - $\Lambda$CDM + $N_\mathrm{eff}$ + $\sum m_\nu$ + $\log T_\mathrm{AGN}$ + $A_\mathrm{kSZ}$ + $n_\mathrm{kSZ}$: additionally vary $A_\mathrm{kSZ}$ and $n_\mathrm{kSZ}$, the amplitude and slope, respectively, of the kSZ power spectrum
  - two CMB lensing noise curves (used in the calculation of the delensed CMB power spectra and the CMB-HD covariance matrix): (1) "MV", the minimum-variance combination of the $TT$, $TE$, $TB$, $EE$, and $EB$ lensing estimators; and (2) "pol-only", using only the $EE$ and $EB$ estimators.
- Table 5: comparison of parameter errors bars in the $\Lambda$CDM + $N_\mathrm{eff}$ + $\sum m_\nu$ + $\log T_\mathrm{AGN}$ + $A_\mathrm{kSZ}$ + $n_\mathrm{kSZ}$ model using the simulation-based noise curves (with the MV lensing noise), or the previous estimate of [MacInnis and Sehgal (2024)](https://arxiv.org/abs/2405.12220).
- Table 6: comparison of the simulation-based parameter error bars in the $\Lambda$CDM + $N_\mathrm{eff}$ + $\sum m_\nu$ + $\log T_\mathrm{AGN}$ model calculated with and without the $TT$ power spectrum; the MV lensing is used in the former case, and the pol-only lensing is used in the latter.

In [ ]:
# print out a tables of parameter error bars from Fisher:
if fisher_matrices_saved:
    # table 4:
    table4_title = '(Table 4) Sim-based CMB-HD delensed TT, TE, EE, BB + kk with DESI BAO'
    fisher_errors = [] # dictionaries of parameter errors
    column_labels = [] # column labels for each set of errors
    # loop through the different parameter models:
    for baryonic_feedback in [False, True]:
        vary_ksz = [False, True] if baryonic_feedback else [False]
        for ksz in vary_ksz:
            params_label = hdfgclean_params.param_model_label(baryonic_feedback=baryonic_feedback, ksz=ksz, use_latex=use_latex)
            # loop through polarization-only or MV lensing:
            for pol_only_lensing in [False, True]:
                lens_label = hdfgclean_params.lensing_label(pol_only_lensing=pol_only_lensing, use_latex=use_latex)
                label = r"%s, %s" % (params_label, lens_label)
                column_labels.append(label)
                # get the parameter errors from the saved fisher matrix:
                errors = hdfgclean_params.get_fisher_errors(fisher_output_dir=fisher_output_dir, 
                                                            baryonic_feedback=baryonic_feedback, ksz=ksz,
                                                            pol_only_lensing=pol_only_lensing)
                fisher_errors.append(errors.copy())
    hdfgclean_params.print_param_table(fisher_errors, column_labels, use_latex=use_latex, title=table4_title)
    
    # table 5:
    table5_title = '(Table 5) Sim-based vs. previous CMB-HD delensed TT, TE, EE, BB + kk with DESI BAO'
    errors = hdfgclean_params.get_fisher_errors(fisher_output_dir=fisher_output_dir, baryonic_feedback=True, ksz=True)
    previous_errors = hdfgclean_params.get_fisher_errors(baryonic_feedback=True, ksz=True, hd_data_version='v1.1')
    hdfgclean_params.print_param_ratio_table(previous_errors, errors, 'Previous', 'Sim-based', title=table5_title)
    
    # table 6:
    table6_title = '(Table 6) Sim-based CMB-HD + DESI BAO, with vs. without TT'
    errors_with_tt = hdfgclean_params.get_fisher_errors(fisher_output_dir=fisher_output_dir, baryonic_feedback=True)
    errors_without_tt = hdfgclean_params.get_fisher_errors(fisher_output_dir=fisher_output_dir, baryonic_feedback=True, 
                                                           pol_only_lensing=True, spectra=['te', 'ee', 'bb', 'kk'])
    hdfgclean_params.print_param_ratio_table(errors_without_tt, errors_with_tt, 'No TT', 'With TT', title=table6_title)

Below, we will plot Figures 12 and 13 of MacInnis et. al. (2026).

- Figure 12 compares the simulation-based parameter forecasts with the previous forecasts of MacInnis and Sehgal (2024).
- Figure 13 compares the simulation-based parameter forecasts using different combinations of CMB-HD and DESI BAO data.

Note that it may take a few minutes to make the plots.

In [ ]:
if fisher_matrices_saved:
    fig12_fname = os.path.join(hdfgcleanlib.plots_dir(), 'figure12.pdf') if save_param_plots else None
    hdfgclean_params.plot_fig12(fisher_output_dir=fisher_output_dir, fname=fig12_fname)

In [ ]:
if fisher_matrices_saved:
    fig13_fname = os.path.join(hdfgcleanlib.plots_dir(), 'figure13.pdf') if save_param_plots else None
    hdfgclean_params.plot_fig13(fisher_output_dir=fisher_output_dir, fname=fig13_fname)